In [1]:
# =============================================================================
# Predicting Irrigation Need — Day 1 Baseline
# Kaggle Playground Series S6E4 | Author: Dean Wang (deanbatur)
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, roc_auc_score
import lightgbm as lgb
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# --- 1. LOAD DATA ---
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv')
sub = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv')

print(f"Train: {train.shape} | Test: {test.shape}")
print(f"Columns: {train.columns.tolist()}")

target_col = sub.columns[-1]
id_col = sub.columns[0]
print(f"\nTarget: {target_col} | ID: {id_col}")
print(f"\nTarget values: {train[target_col].unique()}")
print(f"Target distribution:\n{train[target_col].value_counts()}")
print(f"\nMissing values: {train.isnull().sum().sum()}")
print(f"\nDtypes:\n{train.dtypes}")

# --- 2. ENCODE TARGET (categorical -> numeric) ---
target_le = LabelEncoder()
target_le.fit(pd.concat([train[target_col], sub[target_col]]).astype(str))
train[target_col] = target_le.transform(train[target_col].astype(str))
print(f"\nTarget classes: {list(target_le.classes_)}")
print(f"Encoded as:    {list(range(len(target_le.classes_)))}")

# --- 3. FEATURE ENGINEERING ---
feature_cols = [c for c in train.columns if c not in [id_col, target_col]]
cat_cols = train[feature_cols].select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = train[feature_cols].select_dtypes(include=['number']).columns.tolist()
print(f"\nCategorical ({len(cat_cols)}): {cat_cols}")
print(f"Numerical   ({len(num_cols)}): {num_cols}")

# Label encode categorical features
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

# Interaction features (top 5 numeric pairs)
if len(num_cols) >= 2:
    for i in range(min(len(num_cols), 5)):
        for j in range(i+1, min(len(num_cols), 5)):
            c1, c2 = num_cols[i], num_cols[j]
            train[f'{c1}_x_{c2}'] = train[c1] * train[c2]
            test[f'{c1}_x_{c2}'] = test[c1] * test[c2]

feature_cols = [c for c in train.columns if c not in [id_col, target_col]]
print(f"Total features: {len(feature_cols)}")

# --- 4. PREPARE ---
X = train[feature_cols].values
y = train[target_col].values
X_test = test[feature_cols].values
test_ids = test[id_col].values

n_classes = len(np.unique(y))
is_binary = n_classes == 2
print(f"\nClasses: {n_classes} ({'binary' if is_binary else 'multiclass'})")

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# --- 5. LIGHTGBM ---
lgb_oof = np.zeros((len(X), n_classes)) if not is_binary else np.zeros(len(X))
lgb_preds = np.zeros((len(X_test), n_classes)) if not is_binary else np.zeros(len(X_test))
lgb_scores = []

lgb_params = {
    'objective': 'binary' if is_binary else 'multiclass',
    'metric': 'binary_logloss' if is_binary else 'multi_logloss',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'max_depth': -1,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'lambda_l1': 0.1,
    'lambda_l2': 0.1,
    'verbosity': -1,
    'n_jobs': -1,
    'random_state': 42,
}
if not is_binary:
    lgb_params['num_class'] = n_classes

print("\n=== LightGBM ===")
lgb_model = None
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]
    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)
    lgb_model = lgb.train(
        lgb_params, dtrain, num_boost_round=2000, valid_sets=[dval],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(500)]
    )
    if is_binary:
        lgb_oof[val_idx] = lgb_model.predict(X_val)
        lgb_preds += lgb_model.predict(X_test) / N_FOLDS
        score = roc_auc_score(y_val, lgb_oof[val_idx])
    else:
        lgb_oof[val_idx] = lgb_model.predict(X_val)
        lgb_preds += lgb_model.predict(X_test) / N_FOLDS
        score = accuracy_score(y_val, lgb_oof[val_idx].argmax(axis=1))
    lgb_scores.append(score)
    print(f"  Fold {fold+1}: {score:.5f}")
print(f"LightGBM CV: {np.mean(lgb_scores):.5f} +/- {np.std(lgb_scores):.5f}")

# --- 6. XGBOOST ---
xgb_preds = np.zeros((len(X_test), n_classes)) if not is_binary else np.zeros(len(X_test))
xgb_scores = []

print("\n=== XGBoost ===")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]
    xgb_model = xgb.XGBClassifier(
        n_estimators=2000, learning_rate=0.05, max_depth=6,
        min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=0.1,
        objective='binary:logistic' if is_binary else 'multi:softprob',
        eval_metric='logloss' if is_binary else 'mlogloss',
        num_class=n_classes if not is_binary else None,
        tree_method='hist',
        random_state=42, n_jobs=-1, early_stopping_rounds=50, verbosity=0
    )
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    if is_binary:
        val_pred = xgb_model.predict_proba(X_val)[:, 1]
        xgb_preds += xgb_model.predict_proba(X_test)[:, 1] / N_FOLDS
        score = roc_auc_score(y_val, val_pred)
    else:
        val_pred = xgb_model.predict(X_val)
        xgb_preds += xgb_model.predict_proba(X_test) / N_FOLDS
        score = accuracy_score(y_val, val_pred)
    xgb_scores.append(score)
    print(f"  Fold {fold+1}: {score:.5f}")
print(f"XGBoost CV: {np.mean(xgb_scores):.5f} +/- {np.std(xgb_scores):.5f}")

# --- 7. FEATURE IMPORTANCE ---
feat_imp = pd.DataFrame({
    'feature': feature_cols,
    'importance': lgb_model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)
print(f"\nTop 10 features:\n{feat_imp.head(10).to_string(index=False)}")

# --- 8. ENSEMBLE & SUBMIT ---
if is_binary:
    ensemble = 0.5 * lgb_preds + 0.5 * xgb_preds
    final_preds = (ensemble > 0.5).astype(int)
else:
    ensemble_probs = 0.5 * lgb_preds + 0.5 * xgb_preds
    final_preds = ensemble_probs.argmax(axis=1)

submission = pd.DataFrame({id_col: test_ids, target_col: final_preds})
submission[target_col] = target_le.inverse_transform(submission[target_col].astype(int))

submission.to_csv('submission.csv', index=False)
print(f"\nSubmission saved! Shape: {submission.shape}")
print(submission.head(10))
print(f"\nPrediction distribution:\n{submission[target_col].value_counts(normalize=True)}")
print("\nDone! Click 'Submit' on submission.csv")

Train: (630000, 21) | Test: (270000, 20)
Columns: ['id', 'Soil_Type', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Field_Area_hectare', 'Mulching_Used', 'Previous_Irrigation_mm', 'Region', 'Irrigation_Need']

Target: Irrigation_Need | ID: id

Target values: ['Low' 'Medium' 'High']
Target distribution:
Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64

Missing values: 0

Dtypes:
id                           int64
Soil_Type                   object
Soil_pH                    float64
Soil_Moisture              float64
Organic_Carbon             float64
Electrical_Conductivity    float64
Temperature_C              float64
Humidity                   float64
Rainfall_mm                float64
Sunlight_Hours             float64
Wind_Speed_kmh             float64
Cro